In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
from diffusers import Kandinsky6SRPipeline
from diffusers.utils import encode_video, load_video

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "dev" / "exports" / "diffusers").exists():
    REPO = REPO.parent
if not (REPO / "dev" / "exports" / "diffusers").exists():
    raise RuntimeError("Run this notebook from the K6 repository or one of its subdirectories.")

In [ ]:
MODEL_REPO = "kandinskylab/Kandinsky-6.0-VSR-5s-Diffusers"
GPU_ID = 0

pipe = Kandinsky6SRPipeline.from_pretrained(MODEL_REPO, dtype=torch.bfloat16)
pipe.enable_model_cpu_offload(gpu_id=GPU_ID)

In [ ]:
BS = 2
INPUT_VIDEO = os.path.join(REPO, "assets/vsr_input.mp4")
OUTPUT_VIDEOS = [
    os.path.join(REPO, f"outputs/diffusers/k6_vsr/bs{BS}/vsr_x2.25_batchid{i}.mp4")
    for i in range(BS)
]
SEED = 1137
GENERATOR = torch.Generator(device=f"cuda:{GPU_ID}").manual_seed(SEED)

# The pipeline only takes/returns tensors. Decode the video once, align it to
# 1 + 8k frames (the SR model's temporal chunk size), and repeat it BS times.
frames, fps = load_video(INPUT_VIDEO, return_fps=True)
video = torch.from_numpy(np.stack([np.array(frame) for frame in frames])).permute(3, 0, 1, 2)
aligned = 1 + 8 * ((video.shape[1] - 1) // 8)
video_batch = video[:, :aligned].unsqueeze(0).repeat(BS, 1, 1, 1, 1)

# 2.25x means x1.125 pixel pre-upscale followed by the x2 KVAE path.
result = pipe(
    video=video_batch,
    resolution_scale=2.25,
    output_type="pt",
    generator=GENERATOR,
)
print(f"frames: {tuple(result.frames.shape)}")

for sr_frames, output_path in zip(result.frames, OUTPUT_VIDEOS, strict=True):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    encode_video(sr_frames.permute(1, 2, 3, 0), fps=round(fps), output_path=output_path)
    print(f"saved: {output_path}")